In [2]:
    
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.ligsaf_utilities import create_rcf_utilities_system
from lignin_saf.ligsaf_settings import hdo_params, h2_pressure
from lignin_saf.ligsaf_units import HydrodeoxygenationReactor




chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()
BT, WWT, gas_mixer = create_rcf_utilities_system()

rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()



# Hydrodeoxygenation reactions

hydrodeoxygenation_rxn = bst.ParallelReaction([
    bst.Reaction('Propylguaiacol,l + 6Hydrogen,g -> 1propylcyclohexane,l + 2Water,l + Methane,g', reactant='Propylguaiacol', phases='lg',
                    X=1.0, basis='mol'),
    bst.Reaction('1Propylsyringol,l + 8Hydrogen,g -> 1propylcyclohexane,l + 3Water,l + 2Methane,g', reactant='Propylsyringol', phases='lg',
                    X=1.0, basis='mol'),
])

h2_in = bst.Stream(ID = 'Hydrogen_In', Hydrogen = 300 , units = 'kmol/hr', P = h2_pressure, phase = 'g')


dodcane_required = hdo_params['solvent_req']   #m3/kg of lignin oil
dodecane_flow = F.RCF_Monomers.F_mass * dodcane_required
solvent_stream = bst.Stream(ID = 'Dodecane_In', Dodecane = dodecane_flow, units = 'm3/hr', P = 101325, T = 300, phase = 'l')


catalyst_required = hdo_params['catalyst_req']      # kg/kg of lignin oil
catalyst_flow = F.RCF_Monomers.F_mass * catalyst_required
catalyst_stream =  bst.Stream(ID = 'HDO_Catalyst_In', Ni2PSiO2 = catalyst_flow, units = 'kg/hr', phase = 's')


mixer = bst.units.Mixer(ins = (h2_in, F.RCF_Monomers, solvent_stream), rigorous = True)
mixer.simulate()


compressor = bst.units.IsentropicCompressor(ins = mixer-0, P = hdo_params['P'], vle = True)
compressor.simulate()

heater = bst.units.HXutility(ins = compressor-0, T = hdo_params['T'], rigorous= True)
heater.simulate()


HDO = HydrodeoxygenationReactor(ins = (compressor-0, catalyst_stream),
                                T = hdo_params['T'],
                                P = hdo_params['P'],
                                tau = hdo_params['tau'],
                                tau_0 = hdo_params['tau_0'],
                                free_frac = hdo_params['free_frac'],
                                V_max = hdo_params['V_max'],
                                aspect_ratio = hdo_params['aspect_ratio'],
                                reaction_1 = hydrodeoxygenation_rxn)
HDO.simulate()



c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: PUMP101> no pump type available at current power (1.96e+03 hp), head (3.23e+03 ft), kinematic viscosity (6.01e-07 m2/s), and NPSH (3.96 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:409: CostWarning: <SolvolysisReactor: RCF103_S> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: PUMP108> power (0 hp) i

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   1.86e-10 kmol/hr (0.012%)
- temperature 1.84e-03 K (0.00059%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  889
                    O2  219
[2] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  33.4
                    NaOH   15.1
[3] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water    925
                    Sucrose  0.243
                    Extract  7.4
                    Acetate  48.6
                    Ash      1e+03
                    Lignin   156
                    Glucan   238
                    ...      112
[4] Meoh_in  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Methanol  1.26e+03
[5] Water_in_meoh  
    phase: 'l', T: 298.

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Hydrogen_In> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\IPython\core\interactiveshell.py:3579: RuntimeWarning: undocked inlet s45 from H1; s45 is now docked at R1
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:1220: CostWarning: <HydrodeoxygenationReactor: R1> Vertical vessel weight (1.646e+06 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  self._vertical_vessel_design(
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\mo

In [3]:
HDO

HydrodeoxygenationReactor: R1
ins...
[0] s45  from  IsentropicCompressor-K1
    phases: ('g', 'l'), T: 315.98 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen        300
                        Dodecane        0.00427
                    (l) Hydrogen        3.26e-08
                        Dodecane        1.01e+03
                        Propylguaiacol  17.4
                        Propylsyringol  14.7
[1] HDO_Catalyst_In  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.63e+03
outs...
[0] s48  
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen           77.7
                        Methane            46.9
                        Dodecane           0.00406
                    (l) Water              79
                        Hydrogen           3.26e-08
                        propylcyclohexane  32.1
                        Dodecane           961
[1] s49  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Ni2PSiO2  4.6